# Qwen 2.5 from Scratch

This notebook is a teaching-first walkthrough of a tiny Qwen 2.5-style language model built in PyTorch.
Qwen 2.5 builds directly on the Llama architecture, so we reuse the same foundation (RoPE, GQA, SwiGLU, RMSNorm) and focus on the four innovations that set Qwen apart.

If you have not already worked through the **Llama notebook** in this repository, **do that first**.
This notebook assumes you understand RoPE, Grouped Query Attention, SwiGLU, and the Llama training loop.
We will not re-derive those ideas here; instead we focus on the four key upgrades that Qwen 2.5 introduces.

## Audience
- Learners who have already built a Llama-style model.
- Students who want to understand how production models iterate on the Llama baseline.
- Anyone curious about weight tying, selective bias, or long-context scaling tricks.

## Prerequisites
- Comfortable reading the Llama notebook (RoPE, GQA, SwiGLU, RMSNorm).
- Basic PyTorch concepts such as tensors, modules, and autograd.
- Understanding of how attention scores are computed (QK^T / sqrt(d_k)).

## Learning Goals
By the end of this notebook, you should be able to:
1. Explain why Qwen keeps bias in QKV projections but removes it everywhere else (selective bias).
2. Implement weight tying between the embedding table and the output head using `F.linear`.
3. Describe how NTK-aware RoPE interpolation extends context length without retraining.
4. Implement LogN attention scaling and explain why longer sequences need sharper attention.
5. Train a tiny Qwen model on Tiny Shakespeare and generate text from it.

## Outline

1. Set up the environment and imports.
2. Define the `QwenConfig` (note the new fields: `tie_embeddings`, `max_train_len`).
3. Build the model one component at a time:
   - RMSNorm (brief recap),
   - RoPE with NTK-aware interpolation (the long-context upgrade),
   - QwenAttention (selective bias + LogN scaling -- the big teaching moments),
   - QwenSwiGLU (same as Llama, brief recap),
   - QwenBlock.
4. Assemble the full Qwen model (highlight: tied embeddings via `F.linear`).
5. Model smoke test (verify: tied weights, selective bias, shapes, loss, gradients).
6. Data pipeline (same as previous notebooks).
7. Data smoke test.
8. Training utilities (same pattern, adapted for Qwen).
9. Lightweight pipeline preview.
10. Text generation.
11. Optional checkpoint demo.
12. Exercise.
13. Common pitfalls and next steps.

## 1. Setup

This install cell keeps the notebook easy to run in a fresh environment such as Colab.
If your environment already has these packages, rerunning this cell is harmless.

In [ ]:
!pip install torch numpy requests -q

## 2. Imports

We gather the core tools once so every later cell can stay focused on one idea.

Teaching note:
- `dataclass` gives us a clean config object.
- `Path` helps keep file paths portable between local Jupyter and Colab-style environments.
- `torch.nn` holds reusable neural network layers.
- `torch.nn.functional` contains tensor-level math functions like `softmax`, `cross_entropy`, and crucially `F.linear` which we will use for weight-tied output projection.

In [ ]:
import math
import os
import time
from dataclasses import dataclass
from pathlib import Path

import requests
import torch
import torch.nn as nn
import torch.nn.functional as F

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 3. Configuration

### `QwenConfig`

This dataclass collects the high-level knobs that control the model.
If you have seen the Llama config, most fields are familiar. Two new fields deserve attention:

- **`tie_embeddings`**: when `True`, the input embedding table and the output projection share the same weight matrix. This is Qwen's default and we will explore its consequences in detail.
- **`max_train_len`**: the sequence length the model was trained on. This matters for two scaling mechanisms: NTK-aware RoPE interpolation (which kicks in when `seq_len > max_train_len`) and LogN attention scaling (which uses the ratio `log(current_len) / log(max_train_len)` to keep attention sharp at longer contexts).

All other fields match Llama: `vocab_size=256` for byte-level tokenization, `n_embd=64` for a tiny teaching model, `n_kv_head=2` for GQA, `rope_theta=10000.0` for RoPE, and `dropout=0.0` for determinism.

In [ ]:
@dataclass
class QwenConfig:
    vocab_size: int = 256        # byte-level tokenization (same as base model)
    n_embd: int = 64             # embedding dimension
    n_head: int = 4              # number of query heads
    n_kv_head: int = 2           # number of key/value heads (GQA: n_kv_head < n_head)
    n_layer: int = 4             # number of transformer blocks
    seq_len: int = 128           # maximum sequence length
    dropout: float = 0.0         # dropout rate
    rope_theta: float = 10000.0  # RoPE base frequency
    tie_embeddings: bool = True  # NEW: share weights between wte and lm_head
    max_train_len: int = 128     # NEW: training sequence length (for LogN + NTK scaling)

config = QwenConfig()
print(config)

## 4. Shared Building Blocks (Brief Recap)

The next two sections reuse components from the Llama notebook.
We copy them here so the notebook is fully self-contained, but we keep explanations brief since they were covered in depth before.

### `RMSNorm`

`RMSNorm` rescales a vector by its root-mean-square magnitude:

`RMSNorm(x) = x / sqrt(mean(x^2) + eps) * gamma`

This is identical to the Llama and GPT-2 notebooks.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        return (x / rms) * self.weight

### RoPE (Rotary Position Embeddings)

RoPE encodes position by rotating query and key vectors instead of adding learned position vectors.
The two functions below are identical to the Llama notebook.

**`precompute_rope_freqs`** builds cos/sin lookup tables.
**`apply_rope`** rotates pairs of dimensions in Q and K using those tables.

We will extend `precompute_rope_freqs` with NTK-aware interpolation in the next section, but first we need the base version as a foundation.

In [ ]:
def precompute_rope_freqs(head_dim: int, seq_len: int, theta: float = 10000.0):
    """Precompute the complex exponentials for RoPE.

    Returns cos and sin tensors of shape (seq_len, head_dim // 2).
    """
    # Frequencies for each dimension pair: theta^(-2i/d) for i in [0, d/2)
    freqs = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
    # Outer product with positions gives the rotation angles
    t = torch.arange(seq_len).float()
    angles = torch.outer(t, freqs)  # (seq_len, head_dim // 2)
    return torch.cos(angles), torch.sin(angles)


def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """Apply rotary embeddings to a tensor.

    Args:
        x: shape (batch, n_head, seq_len, head_dim)
        cos, sin: shape (seq_len, head_dim // 2), precomputed frequencies

    The rotation is applied to pairs of dimensions:
        [x0, x1] -> [x0 * cos - x1 * sin, x0 * sin + x1 * cos]
    """
    T = x.shape[2]
    x_pairs = x.float().reshape(*x.shape[:-1], -1, 2)
    x0 = x_pairs[..., 0]  # even dimensions
    x1 = x_pairs[..., 1]  # odd dimensions

    cos_t = cos[:T].unsqueeze(0).unsqueeze(0)  # (1, 1, T, head_dim//2)
    sin_t = sin[:T].unsqueeze(0).unsqueeze(0)  # (1, 1, T, head_dim//2)

    out0 = x0 * cos_t - x1 * sin_t
    out1 = x0 * sin_t + x1 * cos_t

    out = torch.stack([out0, out1], dim=-1).flatten(-2)
    return out.type_as(x)

## 5. Qwen-Specific Components

Now we arrive at the heart of this notebook: the four innovations that distinguish Qwen 2.5 from Llama.

### NTK-aware RoPE Interpolation

#### The problem with standard RoPE at long contexts

Standard RoPE is trained with a fixed `max_train_len`. If you try to use the model at a longer sequence length, the rotation angles go into territory the model has never seen, and quality degrades rapidly.

A naive fix is **linear interpolation**: scale all positions down by `seq_len / max_train_len` so they fit in the original range. But this compresses high-frequency components and hurts local attention patterns.

#### The NTK-aware solution

NTK-aware interpolation takes a smarter approach inspired by Neural Tangent Kernel theory. Instead of scaling positions, it scales the **RoPE base frequency**:

```
alpha = some scaling factor (e.g., related to seq_len / max_train_len)
base_scaled = base * ((alpha * seq_len / max_train_len - (alpha-1)) ** (dim / (dim-2)))
```

The key insight is that different frequency bands need different treatment:
- **High-frequency** dimensions (local patterns): should NOT be compressed much.
- **Low-frequency** dimensions (global patterns): can be safely stretched.

By scaling the base, NTK-aware interpolation naturally achieves this: it stretches low frequencies more than high frequencies. Think of it as "zooming out the clock" -- the slow-ticking dimensions get even slower (handling longer range), while the fast-ticking dimensions stay mostly untouched (preserving local attention).

When `seq_len <= max_train_len`, the scaling factor is 1.0 and we get standard RoPE.

In [ ]:
def precompute_rope_freqs_ntk(
    head_dim: int,
    seq_len: int,
    max_train_len: int,
    theta: float = 10000.0,
    alpha: float = 1.0,
):
    """Precompute RoPE frequencies with NTK-aware interpolation.

    When seq_len > max_train_len, the base frequency is scaled up so that
    low-frequency (global) components stretch to cover the longer range
    while high-frequency (local) components are minimally affected.

    When seq_len <= max_train_len, this is identical to standard RoPE.
    """
    base = theta

    # NTK-aware scaling: only activate when we exceed training length
    if seq_len > max_train_len:
        # Scale the base frequency to handle the longer context
        # The exponent dim/(dim-2) ensures the scaling affects the full spectrum
        scale_factor = alpha * seq_len / max_train_len - (alpha - 1)
        base = base * (scale_factor ** (head_dim / (head_dim - 2)))

    # Standard RoPE computation with (possibly scaled) base
    freqs = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    t = torch.arange(seq_len).float()
    angles = torch.outer(t, freqs)  # (seq_len, head_dim // 2)
    return torch.cos(angles), torch.sin(angles)

### `QwenAttention`

This is the **big teaching moment** of the notebook. Qwen's attention layer has two innovations layered on top of Llama's GQA:

#### Innovation 1: Selective Bias

Most modern transformers (GPT-2, Llama, Mistral) removed all bias terms from linear layers. This simplifies the architecture and reduces parameter count. Qwen 2.5 goes against this trend -- but only partially.

Qwen keeps `bias=True` **only** in the QKV projections (W_q, W_k, W_v) and uses `bias=False` everywhere else (W_o, MLP layers, lm_head).

Why? The Qwen team found that bias in QKV projections helps with **context length extrapolation**. The intuition: bias terms in QKV give the model a position-independent offset for queries and keys. This baseline offset helps the attention mechanism generalize to sequence lengths beyond what it was trained on. Without bias, the attention pattern depends entirely on learned position-dependent features, which may not extrapolate as well.

It is a subtle design choice: just 3 small bias vectors per layer, but they measurably help long-context performance.

#### Innovation 2: LogN Attention Scaling

Standard attention computes: `softmax(QK^T / sqrt(d_k))`

Qwen adds an extra scaling factor: `softmax(QK^T / sqrt(d_k) * logn_scale)`

Where: `logn_scale = log(current_seq_len) / log(max_train_len)`

Why? As sequence length grows, the softmax attention distribution spreads thinner across more tokens (entropy increases). This means each token gets less attention weight, and the signal-to-noise ratio drops. LogN scaling compensates by making attention slightly sharper for longer sequences.

At training length, `log(T) / log(T) = 1.0`, so the factor has no effect. At 2x training length, the factor is `log(2T) / log(T) > 1.0`, which sharpens attention. At 0.5x training length, the factor is `< 1.0`, which softens attention. This creates a natural, principled adaptive mechanism.

The `log` function is chosen because attention entropy grows roughly logarithmically with sequence length.

In [ ]:
class QwenAttention(nn.Module):
    def __init__(self, config: QwenConfig, cos: torch.Tensor, sin: torch.Tensor):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        assert config.n_head % config.n_kv_head == 0

        self.n_head = config.n_head
        self.n_kv_head = config.n_kv_head
        self.n_groups = config.n_head // config.n_kv_head  # queries per KV group
        self.head_dim = config.n_embd // config.n_head
        self.max_train_len = config.max_train_len

        # ---------------------------------------------------------------
        # SELECTIVE BIAS: bias=True ONLY for QKV, bias=False for W_o
        # ---------------------------------------------------------------
        # This is Qwen's signature design choice. Compare to Llama where
        # ALL linear layers use bias=False.
        self.W_q = nn.Linear(config.n_embd, config.n_head * self.head_dim, bias=True)      # <-- bias=True!
        self.W_k = nn.Linear(config.n_embd, config.n_kv_head * self.head_dim, bias=True)   # <-- bias=True!
        self.W_v = nn.Linear(config.n_embd, config.n_kv_head * self.head_dim, bias=True)   # <-- bias=True!
        self.W_o = nn.Linear(config.n_embd, config.n_embd, bias=False)                     # <-- bias=False

        self.dropout = nn.Dropout(config.dropout)

        # RoPE frequencies (registered as buffers so they move to the right device)
        self.register_buffer("rope_cos", cos)
        self.register_buffer("rope_sin", sin)

        # Causal mask
        causal_mask = torch.tril(torch.ones(config.seq_len, config.seq_len))
        self.register_buffer("causal_mask", causal_mask.view(1, 1, config.seq_len, config.seq_len))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape

        # Project to Q, K, V (with bias!)
        q = self.W_q(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.W_k(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.W_v(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)

        # Apply RoPE to Q and K (not V -- values don't need position info)
        q = apply_rope(q, self.rope_cos, self.rope_sin)
        k = apply_rope(k, self.rope_cos, self.rope_sin)

        # Expand K,V to match Q heads by repeating each KV head n_groups times
        if self.n_groups > 1:
            k = k.repeat_interleave(self.n_groups, dim=1)
            v = v.repeat_interleave(self.n_groups, dim=1)

        # Compute attention scores
        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        # ---------------------------------------------------------------
        # LOGN ATTENTION SCALING
        # ---------------------------------------------------------------
        # Scale attention scores based on current vs training sequence length.
        # At training length: factor = 1.0 (no effect).
        # At longer lengths: factor > 1.0 (sharpens attention).
        # This compensates for entropy increase in longer sequences.
        logn_scale = math.log(T) / math.log(self.max_train_len) if T > 1 else 1.0
        attn = attn * logn_scale

        # Apply causal mask and softmax
        attn = attn.masked_fill(self.causal_mask[:, :, :T, :T] == 0, float("-inf"))
        attn = F.softmax(attn, dim=-1)
        attn = self.dropout(attn)

        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.W_o(out)  # Output projection (no bias)
        return out

### `QwenSwiGLU` MLP

This is identical to the Llama SwiGLU MLP. We keep the recap brief.

The gated MLP does:
`SwiGLU(x) = Swish(W_gate * x) * (W_up * x)`

Then projects back down:
`output = W_down * SwiGLU(x)`

The hidden dimension is `4 * n_embd * 2/3`, rounded to the nearest multiple of 8.

Note: all three linear layers use `bias=False`, consistent with Qwen's selective bias strategy (bias only in QKV).

In [ ]:
class QwenSwiGLU(nn.Module):
    def __init__(self, config: QwenConfig):
        super().__init__()
        # Hidden dim adjusted to keep param count ~same as 4x expansion
        hidden_dim = int(4 * config.n_embd * 2 / 3)
        # Round to nearest multiple of 8 for hardware efficiency
        hidden_dim = ((hidden_dim + 7) // 8) * 8

        self.w_gate = nn.Linear(config.n_embd, hidden_dim, bias=False)  # gate (no bias!)
        self.w_up = nn.Linear(config.n_embd, hidden_dim, bias=False)    # up-projection (no bias!)
        self.w_down = nn.Linear(hidden_dim, config.n_embd, bias=False)  # down-projection (no bias!)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gate = F.silu(self.w_gate(x))  # Swish = SiLU = x * sigmoid(x)
        up = self.w_up(x)
        x = gate * up
        x = self.w_down(x)
        x = self.dropout(x)
        return x

### `QwenBlock`

A Qwen block is structurally identical to a Llama block: two sublayers with residual connections and pre-norm.
1. Attention (now `QwenAttention` with selective bias + LogN scaling),
2. MLP (QwenSwiGLU, same as Llama but with explicit `bias=False`).

The pre-norm pattern and residual connections are unchanged.

In [ ]:
class QwenBlock(nn.Module):
    def __init__(self, config: QwenConfig, cos: torch.Tensor, sin: torch.Tensor):
        super().__init__()
        self.ln_1 = RMSNorm(config.n_embd)
        self.attn = QwenAttention(config, cos, sin)
        self.ln_2 = RMSNorm(config.n_embd)
        self.mlp = QwenSwiGLU(config)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

## 6. Full Qwen Model

### `Qwen`

This class assembles the full language model. Compare it to Llama and notice the **one big structural difference**:

**Tied embeddings: the input embedding table and output projection share the same weight matrix.**

In Llama (and GPT-2), the model has two separate matrices:
- `wte.weight` of shape `(vocab_size, n_embd)` -- maps token IDs to vectors.
- `lm_head.weight` of shape `(vocab_size, n_embd)` -- maps vectors back to token logits.

In Qwen, these are the **same matrix**. The embedding table maps token -> vector, and the output head maps vector -> token. If both use the same matrix, tokens that embed similarly will also decode similarly. This creates a powerful symmetry.

Implementation detail: we do NOT create a separate `nn.Linear` for `lm_head`. Instead, in the forward pass, we use `F.linear(x, self.wte.weight)` to multiply by the transposed embedding matrix. This saves `vocab_size * n_embd` parameters.

For our tiny model with `vocab_size=256` and `n_embd=64`, that saves 16,384 parameters. In production Qwen models with `vocab_size=151,936` and `n_embd=4,096`, weight tying saves over 600 million parameters.

When `tie_embeddings=False`, we fall back to a separate `lm_head` like Llama, so you can compare both approaches.

In [ ]:
class Qwen(nn.Module):
    def __init__(self, config: QwenConfig):
        super().__init__()
        self.config = config

        # Token embedding only -- no positional embedding! (RoPE handles position)
        self.wte = nn.Embedding(config.vocab_size, config.n_embd)

        # Precompute RoPE frequencies with NTK-aware interpolation
        head_dim = config.n_embd // config.n_head
        cos, sin = precompute_rope_freqs_ntk(
            head_dim, config.seq_len, config.max_train_len, config.rope_theta
        )

        # Stack of transformer blocks
        self.blocks = nn.ModuleList([
            QwenBlock(config, cos, sin) for _ in range(config.n_layer)
        ])

        self.ln_f = RMSNorm(config.n_embd)

        # -------------------------------------------------------------------
        # TIED EMBEDDINGS: conditionally create a separate lm_head
        # -------------------------------------------------------------------
        # When tie_embeddings=True (Qwen default): no separate lm_head.
        # The forward pass uses F.linear(x, self.wte.weight) instead.
        # When tie_embeddings=False: create a separate lm_head like Llama.
        if not config.tie_embeddings:
            self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        # Initialize weights
        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("W_o.weight") or name.endswith("w_down.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            # Initialize bias if present (only QKV projections in Qwen)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.config.seq_len, f"Sequence length {T} exceeds max {self.config.seq_len}"

        # Only token embeddings -- RoPE adds position info inside attention
        x = self.wte(idx)  # (B, T, n_embd)

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)

        # -------------------------------------------------------------------
        # OUTPUT PROJECTION: use tied weights or separate lm_head
        # -------------------------------------------------------------------
        if self.config.tie_embeddings:
            # F.linear(input, weight) computes input @ weight.T
            # This is equivalent to nn.Linear but using the embedding weight directly.
            # No separate parameter matrix needed!
            logits = F.linear(x, self.wte.weight)
        else:
            logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))

        return logits, loss

    def count_parameters(self):
        # When tie_embeddings=True, we need to count unique parameters
        # (wte.weight is shared, so it should only be counted once)
        total = sum(p.numel() for p in self.parameters())
        print(f"Total parameters: {total:,}")
        print(f"  Token embeddings (wte): {self.wte.weight.numel():,}")
        for i, block in enumerate(self.blocks):
            block_params = sum(p.numel() for p in block.parameters())
            print(f"  Block {i}: {block_params:,}")
        print(f"  Final norm: {sum(p.numel() for p in self.ln_f.parameters()):,}")
        if self.config.tie_embeddings:
            print(f"  LM head:   TIED to wte (0 extra parameters)")
        else:
            print(f"  LM head:   {self.lm_head.weight.numel():,}")
        return total

## 7. Model Smoke Test

Before training anything, we want a quick confidence check that the tensor shapes, gradients, and Qwen-specific features behave as expected.

This test verifies five things:
1. **Tied embeddings**: the model has no separate `lm_head` attribute when `tie_embeddings=True`.
2. **Selective bias**: QKV projections have bias, W_o does not.
3. **Shapes**: logits come out with shape `(batch, time, vocab_size)`.
4. **Loss**: random-initialization loss is near `log(256) = 5.55`.
5. **Parameter count comparison**: tied vs untied to see the savings.

In [ ]:
model = Qwen(config)
model.count_parameters()

# --- 1. Verify tied embeddings ---
if config.tie_embeddings:
    assert not hasattr(model, "lm_head"), "With tie_embeddings=True, there should be no separate lm_head"
    print("\nTied embeddings: no separate lm_head attribute (output uses F.linear with wte.weight)")
else:
    print("\nUntied embeddings: separate lm_head exists")

# --- 2. Verify selective bias: QKV have bias, W_o does not ---
block0 = model.blocks[0].attn
assert block0.W_q.bias is not None, "W_q should have bias"
assert block0.W_k.bias is not None, "W_k should have bias"
assert block0.W_v.bias is not None, "W_v should have bias"
assert block0.W_o.bias is None, "W_o should NOT have bias"
print("Selective bias: W_q/W_k/W_v have bias=True, W_o has bias=False")

# --- 3. Forward pass shape check ---
idx = torch.randint(0, config.vocab_size, (2, 32))
logits, _ = model(idx)
assert logits.shape == (2, 32, config.vocab_size), f"Logit shape wrong: {logits.shape}"
print(f"\nForward pass: idx {idx.shape} -> logits {logits.shape}")

# --- 4. Loss near log(256) ---
targets = torch.randint(0, config.vocab_size, (2, 32))
logits, loss = model(idx, targets)
expected_loss = math.log(config.vocab_size)
print(f"Random-init loss: {loss.item():.4f} (expected ~{expected_loss:.4f})")
assert abs(loss.item() - expected_loss) < 1.0, "Loss too far from expected"

# --- 5. Gradient flow ---
loss.backward()
grad_count = sum(1 for p in model.parameters() if p.grad is not None)
print(f"Parameters with gradients: {grad_count}")

# --- 6. Parameter count comparison: tied vs untied ---
tied_params = sum(p.numel() for p in model.parameters())
untied_config = QwenConfig(tie_embeddings=False)
untied_model = Qwen(untied_config)
untied_params = sum(p.numel() for p in untied_model.parameters())
savings = untied_params - tied_params
print(f"\nParameter comparison:")
print(f"  Tied:   {tied_params:,}")
print(f"  Untied: {untied_params:,}")
print(f"  Savings from tying: {savings:,} ({savings / untied_params * 100:.1f}%)")

## 8. Data Pipeline

A language model learns from one very long stream of tokens.
Our job is to turn that stream into many smaller `(input, target)` examples.

This pipeline is identical to the GPT-2 and Llama notebooks: raw UTF-8 bytes as tokens, Tiny Shakespeare as the dataset, and random windowed sampling for batches.

### Dataset Paths and Constants

This cell finds the project root in a way that works both when the notebook is opened from the repository root and when it is opened from the `notebook/` folder.

Checkpoints for this Qwen notebook are saved to a `qwen` subdirectory to keep them separate from other architecture checkpoints.

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "model.py").exists() and (PROJECT_ROOT.parent / "model.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints" / "qwen"
SHAKESPEARE_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Checkpoint directory: {CHECKPOINT_DIR}")

In [ ]:
def encode(text: str) -> list[int]:
    return list(text.encode("utf-8"))


def decode(tokens: list[int]) -> str:
    return bytes(tokens).decode("utf-8", errors="replace")


def download_shakespeare() -> str:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    filepath = DATA_DIR / "shakespeare.txt"

    if not filepath.exists():
        print(f"Downloading tiny Shakespeare to {filepath}...")
        response = requests.get(SHAKESPEARE_URL)
        response.raise_for_status()
        filepath.write_text(response.text)
        print(f"Downloaded {len(response.text):,} characters.")

    return filepath.read_text()


def get_batch(data: torch.Tensor, batch_size: int, seq_len: int, device: str = "cpu"):
    max_start = len(data) - seq_len - 1
    starts = torch.randint(0, max_start, (batch_size,))
    x = torch.stack([data[i : i + seq_len] for i in starts])
    y = torch.stack([data[i + 1 : i + 1 + seq_len] for i in starts])
    return x.to(device), y.to(device)


def prepare_data(val_fraction: float = 0.1):
    text = download_shakespeare()
    tokens = encode(text)
    data = torch.tensor(tokens, dtype=torch.long)
    split_idx = int(len(data) * (1 - val_fraction))
    train_data = data[:split_idx]
    val_data = data[split_idx:]
    return train_data, val_data

## 9. Data Smoke Test

We test tokenizer round-trip correctness, dataset size, and the shift-by-one behavior in training batches.

In [ ]:
sample_text = "Hello, World!"
sample_tokens = encode(sample_text)
round_trip = decode(sample_tokens)
print(sample_tokens)
print(round_trip)
assert round_trip == sample_text

train_data, val_data = prepare_data()
print(f"Total tokens: {len(train_data) + len(val_data):,}")
print(f"Train tokens: {len(train_data):,}")
print(f"Val tokens:   {len(val_data):,}")

x, y = get_batch(train_data, batch_size=4, seq_len=32)
print(f"Batch shapes: x={x.shape}, y={y.shape}")
print(f"First input preview:  {decode(x[0].tolist())!r}")
print(f"First target preview: {decode(y[0].tolist())!r}")
assert torch.equal(x[0, 1:], y[0, :-1])

## 10. Training Utilities

These helpers make the training loop readable. Each introduces one concept at a time.

In [ ]:
def get_lr(step: int, max_lr: float, min_lr: float, warmup_steps: int, total_steps: int) -> float:
    if step < warmup_steps:
        return max_lr * (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    cosine = 0.5 * (1 + math.cos(math.pi * progress))
    return min_lr + (max_lr - min_lr) * cosine


@torch.no_grad()
def estimate_loss(model, train_data, val_data, batch_size, seq_len, device, eval_iters=20):
    model.eval()
    results = {}
    for name, data in [("train", train_data), ("val", val_data)]:
        losses = []
        for _ in range(eval_iters):
            x, y = get_batch(data, batch_size, seq_len, device)
            _, loss = model(x, y)
            losses.append(loss.item())
        results[name] = sum(losses) / len(losses)
    model.train()
    return results


def save_checkpoint(model, config, optimizer, step, path):
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "config": config,
            "step": step,
        },
        path,
    )


def load_checkpoint(path, device="cpu"):
    checkpoint = torch.load(path, map_location=device, weights_only=False)
    config = checkpoint["config"]
    model = Qwen(config).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    return model, config, checkpoint["step"]


def get_device():
    if torch.cuda.is_available():
        return "cuda"
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    return "cpu"

In [ ]:
def sanity_check(model, train_data, val_data, config, device, steps=50):
    """Quick overfitting test: train on one tiny batch and verify loss drops."""
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

    # Fix a single batch so we can memorize it
    x, y = get_batch(train_data, batch_size=4, seq_len=config.seq_len, device=device)

    print("Sanity check: overfitting one batch...")
    initial_loss = None
    for step in range(steps):
        _, loss = model(x, y)
        if initial_loss is None:
            initial_loss = loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if step % 10 == 0:
            print(f"  step {step:3d} | loss {loss.item():.4f}")

    final_loss = loss.item()
    assert final_loss < initial_loss * 0.7, \
        f"Sanity check failed: loss didn't drop enough ({initial_loss:.4f} -> {final_loss:.4f})"
    print(f"Sanity check passed! Loss: {initial_loss:.4f} -> {final_loss:.4f}")
    return model

In [ ]:
def train(
    model,
    train_data,
    val_data,
    config,
    device,
    max_steps=2000,
    batch_size=32,
    max_lr=3e-4,
    min_lr=3e-5,
    warmup_steps=100,
    eval_interval=200,
    checkpoint_interval=500,
):
    """Full training loop with cosine LR schedule and periodic evaluation."""
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=max_lr)
    model.train()

    t0 = time.time()
    for step in range(max_steps):
        # Cosine learning rate schedule
        lr = get_lr(step, max_lr, min_lr, warmup_steps, max_steps)
        for param_group in optimizer.param_groups:
            param_group["lr"] = lr

        # Forward + backward
        x, y = get_batch(train_data, batch_size, config.seq_len, device)
        _, loss = model(x, y)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        # Print progress every 50 steps
        if step % 50 == 0:
            elapsed = time.time() - t0
            tokens_per_sec = (step + 1) * batch_size * config.seq_len / elapsed if elapsed > 0 else 0
            print(f"step {step:5d} | loss {loss.item():.4f} | lr {lr:.2e} | {tokens_per_sec:.0f} tok/s")

        # Periodic evaluation
        if step > 0 and step % eval_interval == 0:
            metrics = estimate_loss(model, train_data, val_data, batch_size, config.seq_len, device)
            print(f"  eval  | train {metrics['train']:.4f} | val {metrics['val']:.4f}")

        # Periodic checkpoint
        if checkpoint_interval and (step > 0 and step % checkpoint_interval == 0):
            path = CHECKPOINT_DIR / f"step_{step}.pt"
            save_checkpoint(model, config, optimizer, step, path)
            print(f"  -> saved checkpoint to {path}")

    # Final checkpoint
    path = CHECKPOINT_DIR / "final.pt"
    save_checkpoint(model, config, optimizer, max_steps, path)
    elapsed = time.time() - t0
    print(f"\nTraining complete in {elapsed:.1f}s. Final checkpoint: {path}")
    return model

## 11. Lightweight Pipeline Preview

Before committing to a full training run, we wire up the entire pipeline end-to-end:
create a fresh model, move it to the best available device, run the sanity check,
and confirm everything works together.

This catches device-mismatch bugs, shape errors, and other integration issues early.

In [ ]:
device = get_device()
print(f"Using device: {device}")

# Fresh model for training
config = QwenConfig()
model = Qwen(config).to(device)

# Prepare data
train_data, val_data = prepare_data()

# Sanity check
model = sanity_check(model, train_data, val_data, config, device)
print("\nPipeline is ready for full training.")

## 12. Train!

Now we train the Qwen model on Tiny Shakespeare. With our tiny config (64-dim, 4 layers), this takes about 1-2 minutes on CPU.

The training loop uses:
- **Cosine LR schedule** with warmup (100 steps warm up, then cosine decay)
- **Gradient clipping** at norm 1.0 for stability
- **Periodic evaluation** on both train and val splits

Tip: increase `max_steps` for better generation quality (try 2000-5000 on GPU).

In [ ]:
# Create a fresh model for the real training run
config = QwenConfig()
model = Qwen(config).to(device)

model = train(
    model, train_data, val_data, config, device,
    max_steps=1000,
    batch_size=32,
    max_lr=3e-4,
    min_lr=3e-5,
    warmup_steps=100,
    eval_interval=200,
    checkpoint_interval=500,
)

## 13. Text Generation

Now for the fun part: generating text from our trained model.

The `generate` function works by repeatedly:
1. Taking the last `seq_len` tokens as context.
2. Running a forward pass to get logits for the next token.
3. Applying temperature scaling (higher = more random, lower = more deterministic).
4. Sampling from the resulting probability distribution.
5. Appending the sampled token and repeating.

In [ ]:
@torch.no_grad()
def generate(model, prompt: str, max_new_tokens: int = 200, temperature: float = 0.8):
    """Generate text from a prompt using the trained model."""
    model.eval()
    device = next(model.parameters()).device
    tokens = encode(prompt)
    idx = torch.tensor([tokens], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        # Crop to max sequence length
        idx_crop = idx[:, -model.config.seq_len :]
        logits, _ = model(idx_crop)

        # Take logits for the last position and apply temperature
        logits = logits[:, -1, :] / temperature

        # Sample from the distribution
        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_token], dim=1)

    return decode(idx[0].tolist())

In [ ]:
# Generate samples with different temperatures
for temp in [0.5, 0.8, 1.0]:
    print(f"\n{'='*60}")
    print(f"Temperature: {temp}")
    print(f"{'='*60}")
    text = generate(model, "ROMEO:", max_new_tokens=200, temperature=temp)
    print(text)

## 14. Checkpoint Demo

We can reload the model from a saved checkpoint and generate text.
This proves the save/load cycle preserves the model state correctly.

In [ ]:
# Load from checkpoint and verify it generates the same quality text
checkpoint_path = CHECKPOINT_DIR / "final.pt"
if checkpoint_path.exists():
    loaded_model, loaded_config, loaded_step = load_checkpoint(checkpoint_path, device)
    print(f"Loaded checkpoint from step {loaded_step}")
    print(f"Config: {loaded_config}")
    print(f"\nGenerated text from loaded model:")
    print(generate(loaded_model, "JULIET:", max_new_tokens=200, temperature=0.8))
else:
    print(f"No checkpoint found at {checkpoint_path}. Run the training cell first.")

## 15. Exercise: Ablation Study

Now that you have a working Qwen model, try these experiments to build intuition about each innovation:

### Exercise 1: Tied vs Untied Embeddings
Train two models — one with `tie_embeddings=True` (default) and one with `tie_embeddings=False`.
Compare parameter counts, final loss, and generation quality. Does tying hurt or help at this tiny scale?

### Exercise 2: Selective Bias Ablation
Modify `QwenAttention` to use `bias=False` for QKV (like Llama). Does removing bias change the loss curve or generation quality?

### Exercise 3: LogN Scaling
Set `logn_scale = 1.0` (hardcode it) to disable LogN scaling. Train and compare. The effect may be subtle at `seq_len=256` but becomes more pronounced at longer contexts.

Below is a scaffold for Exercise 1:

In [ ]:
# Exercise 1 scaffold: Tied vs Untied Embeddings
#
# Uncomment and run to compare:
#
# tied_config = QwenConfig(tie_embeddings=True)
# untied_config = QwenConfig(tie_embeddings=False)
#
# tied_model = Qwen(tied_config).to(device)
# untied_model = Qwen(untied_config).to(device)
#
# tied_params = sum(p.numel() for p in tied_model.parameters())
# untied_params = sum(p.numel() for p in untied_model.parameters())
# print(f"Tied:   {tied_params:,} params")
# print(f"Untied: {untied_params:,} params")
# print(f"Savings: {untied_params - tied_params:,} params")
#
# # Train both (use fewer steps for quick comparison)
# tied_model = train(tied_model, train_data, val_data, tied_config, device, max_steps=500, checkpoint_interval=0)
# untied_model = train(untied_model, train_data, val_data, untied_config, device, max_steps=500, checkpoint_interval=0)
#
# # Compare generation
# print("\n--- Tied ---")
# print(generate(tied_model, "ROMEO:", max_new_tokens=100))
# print("\n--- Untied ---")
# print(generate(untied_model, "ROMEO:", max_new_tokens=100))

## 16. Common Pitfalls and Next Steps

### Common pitfalls

1. **Forgetting to tie weights at init time.** If you create a separate `nn.Linear` for `lm_head` and then try to assign `lm_head.weight = wte.weight` after `__init__`, the optimizer may not track it correctly. Use `F.linear` in the forward pass instead.

2. **LogN scaling with T=1.** When `T=1`, `log(1) = 0`, which would make the scale factor 0 and kill all attention. Guard against this (we use `if T > 1`).

3. **NTK scaling when seq_len <= max_train_len.** The NTK interpolation should be a no-op at training length. If your base frequency changes when it shouldn't, check your condition.

4. **Selective bias inconsistency.** If you accidentally add bias to W_o or MLP layers, it won't crash — but it's no longer Qwen's architecture, and you lose the long-context benefit.

### What Qwen teaches in the bigger picture

Qwen 2.5 shows that **small, targeted changes** to a proven architecture (Llama) can yield significant improvements:
- Tied embeddings save millions of parameters with minimal quality loss.
- Selective bias is a surgical intervention: 3 tiny vectors per layer for better extrapolation.
- NTK-aware RoPE and LogN scaling are complementary long-context tricks that compose cleanly.

### Next steps

- **MoE notebook**: See how Qwen's base architecture combines with mixture-of-experts routing.
- **Sliding Window notebook**: See how local/global attention patterns improve efficiency.
- **Compare all architectures**: Run `examples/compare_architectures.py` to see GPT-2, Llama, Qwen, MoE, and Sliding Window side by side.
- **Scale up**: Try `n_embd=128`, `n_layer=6` and see how the innovations scale.